**학습 목표**: 같은 데이터·같은 평가지표로 두 앙상블 모델을 비교하고, 성능과 특성중요도의 차이를 관찰할 수 있다.

In [1]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.4/48.9 MB 16.0 MB/s eta 0:00:03
   ----- ---------------------------------- 6.6/48.9 MB 18.5 MB/s eta 0:00:03
   -------- ------------------------------- 10.2/48.9 MB 17.9 MB/s eta 0:00:03
   ----------- ---------------------------- 13.9/48.9 MB 17.9 MB/s eta 0:00:02
   -------------- ------------------------- 18.4/48.9 MB 18.7 MB/s eta 0:00:02
   ------------------ --------------------- 22.8/48.9 MB 19.1 MB/s eta 0:00:02
   --------------------- ------------------ 26.7/48.9 MB 19.2 MB/s eta 0:00:02
   ------------------------- -------------- 31.2/48.9 MB 19.5 MB/s eta 0:00:01
   ----------------------------- ---------- 35.9/48.9 MB 19.7 MB/s eta 0:00:01
   --------------------------------- ------ 40.6/48.9 MB 20.0 MB/s eta 0:00:01
   ------------------------------------- -- 45.6/48.9 MB 20.2 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 20.3 M


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier


In [35]:
def compare_rf_xgb(X: np.ndarray, y: np.ndarray, random_state: int = 42) -> dict:
    """
    요구사항:
    - train_test_split으로 데이터를 나눈다.
    - RandomForestClassifier와 XGBClassifier(또는 xgboost 미설치 시 GradientBoostingClassifier)를
      각각 같은 train 세트로 학습한다.
    - 테스트 세트에서 accuracy와 ROC-AUC를 각각 계산한다.
    - 두 모델의 feature_importances_ 상위 5개를 각각 반환한다.
    - 결과를 dict로 반환 (모델명 → {accuracy, roc_auc, top5_features}).
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
    randomforest_model = RandomForestClassifier()
    XGB_model = XGBClassifier() or GradientBoostingClassifier()
    models = []
    models.append(randomforest_model)
    models.append(XGB_model)
    results = {}
    for model in models:
        model.fit(X_train, y_train)
        model.predict_proba(X_test)[:,1]
        Accuracy = model.score(X_test, y_test)
        ROC_AUC = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        idx = np.argsort(model.feature_importances_)[::-1]
        importances = model.feature_importances_[idx][:5]
        results[model] = {'accuracy' : Accuracy, 'roc_auc' : ROC_AUC, 'importances' : importances}
    
    return results
data = load_breast_cancer()
result = compare_rf_xgb(data.data, data.target)
for model_name, metrics in result.items():
    print(f"\n{model_name}")
    print(f"  accuracy: {metrics['accuracy']:.4f}")
    print(f"  roc_auc : {metrics['roc_auc']:.4f}")
    print(f"  top5   : {metrics['importances']}")



RandomForestClassifier()
  accuracy: 0.9474
  roc_auc : 0.9927
  top5   : [0.14767199 0.14327764 0.12143529 0.10364375 0.10037935]

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)
  accuracy: 0.9591
  roc_auc : 0.9962
  top5   : [0.27524477 0.2451945  0.22447193 0.04652448 0.03781951]
